# SSDM Public Health Dashboard, Colab Launcher

Run the cells in order. Cell 1 clones the repository and installs dependencies. Cell 2 mounts Google Drive and links the data home. Cell 3 checks data freshness and rebuilds the six CDC surveillance sheets when the workbook is older than seven days. Cell 4 loads the Gemini key from Colab Secrets. Cell 5 starts the dashboard and prints its public link.

First run note: the initial CDC rebuild downloads the full feed histories (about 160,000 records) and takes several minutes. Later runs skip it unless the data are stale.

In [ ]:
# Cell 1: clone the repository and install dependencies
import os, subprocess, sys

REPO_URL = "https://github.com/Shayan9574/ssdm-dashboard.git"
REPO_DIR = "/content/ssdm-dashboard"

if os.path.exists(REPO_DIR):
    print(subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True).stdout)
else:
    r = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr)
        from getpass import getpass
        token = getpass("Repository looks private. Paste a GitHub token with read access: ")
        auth_url = REPO_URL.replace("https://", f"https://Shayan9574:{token}@")
        r = subprocess.run(["git", "clone", auth_url, REPO_DIR], capture_output=True, text=True)
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
    print("clone:", "OK" if r.returncode == 0 else r.stderr)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("dependencies installed; working dir:", os.getcwd())

In [ ]:
# Cell 2: mount Google Drive and link the data home
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DATA = Path("/content/drive/MyDrive/SSDM-Dashboard/data")
DRIVE_DATA.mkdir(parents=True, exist_ok=True)
os.environ["SSDM_DATA_DIR"] = str(DRIVE_DATA)

# Link the repository data paths to Drive so every module reads and writes the Drive master.
repo_data = Path("/content/ssdm-dashboard/data")
for name in ["Research Data.xlsx", "pipeline_status.json"]:
    link = repo_data / name
    target = DRIVE_DATA / name
    if link.is_symlink() or link.exists():
        link.unlink()
    link.symlink_to(target)
print("data home on Drive:", DRIVE_DATA)
print("master exists:", (DRIVE_DATA / "Research Data.xlsx").exists())

In [ ]:
# Cell 3: staleness check, rebuild the six CDC feed sheets when older than seven days
import json, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path

FORCE_REFRESH = False   # set True to rebuild regardless of age
MAX_AGE_DAYS = 7

DRIVE_DATA = Path("/content/drive/MyDrive/SSDM-Dashboard/data")
status_file = DRIVE_DATA / "pipeline_status.json"
master = DRIVE_DATA / "Research Data.xlsx"

def age_days():
    if not master.exists() or not status_file.exists():
        return None
    try:
        ts = json.loads(status_file.read_text())["last_sync_timestamp"]
        last = datetime.strptime(ts, "%Y-%m-%d %H:%M UTC").replace(tzinfo=timezone.utc)
        return (datetime.now(timezone.utc) - last).total_seconds() / 86400.0
    except Exception:
        return None

a = age_days()
stale = FORCE_REFRESH or a is None or a > MAX_AGE_DAYS
print(f"workbook age: {'unknown' if a is None else f'{a:.1f} days'}; refresh needed: {stale}")

if stale:
    proc = subprocess.Popen([sys.executable, "scripts/refresh_data.py"],
                            cwd="/content/ssdm-dashboard",
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print("refresh finished with code", proc.returncode)
else:
    print("data are current; no refresh performed")

In [ ]:
# Cell 4: Gemini key from Colab Secrets (Secrets panel on the left, add GEMINI_API_KEY)
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("Gemini key loaded from Colab Secrets")
except Exception as e:
    print("No Gemini key available yet; AI features will be inactive.", e)

In [ ]:
# Cell 5: launch Streamlit and expose it through a Cloudflare tunnel
import os, re, subprocess, time, socket, urllib.request

os.chdir("/content/ssdm-dashboard")
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)

if not os.path.exists("/usr/local/bin/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/usr/local/bin/cloudflared")
    os.chmod("/usr/local/bin/cloudflared", 0o755)

st = subprocess.Popen(["streamlit", "run", "app.py",
                       "--server.port", "8501", "--server.headless", "true",
                       "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                      env={**os.environ})

for _ in range(60):
    try:
        socket.create_connection(("localhost", 8501), timeout=1).close()
        break
    except OSError:
        time.sleep(1)
print("streamlit is up on port 8501")

cf = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 60 and url is None:
    line = cf.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m:
        url = m.group(0)
print("=" * 60)
print("DASHBOARD URL:", url or "tunnel did not report a URL; rerun this cell")
print("=" * 60)
print("Keep this cell running; the dashboard lives as long as this Colab session.")